[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_gradient-check.ipynb)

# Forward-Mode vs. Reverse-Mode Gradients Through the Elastic Solver

Takes the exact same composite RVE and shear-strain setup as
[`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb) and asks a different question: can we
differentiate the solve with respect to a material parameter (here, the fibre's Young's modulus
`E_fiber`), and do forward-mode (`jax.jvp`) and reverse-mode (`jax.grad`) autodiff agree on the
answer?

**Why this needs its own CG implementation, not `solvers.mechanical.strain_nw_cg.solve_elastic`:**
the production solver uses `jax.scipy.sparse.linalg.cg` internally (fast early-exit via
`jax.lax.while_loop`, adopted for speed). That was tested against this project's FFT-based
operators and found to give a **silently wrong** reverse-mode gradient (off by many orders of
magnitude on a heterogeneous composite, NaN on a degenerate homogeneous one) -- `while_loop` has
no reverse-mode rule at all, and `jax.scipy.sparse.linalg.cg`'s own implicit-diff workaround for
that isn't reliable here. Forward-mode (`jax.jvp`) doesn't have this problem either way.

So this notebook defines a small standalone CG (`cg_diff` below) via `jax.lax.scan` over a fixed
number of steps instead of `jax.lax.while_loop` -- `scan`'s trip count is static, so it has a
well-defined reverse-mode rule. The cost is that it always runs the full step budget rather than
stopping early, which is why it's a separate, deliberately-not-used-in-production implementation
kept local to this notebook, not merged back into `solvers.mechanical.strain_nw_cg`.

## Setup

In [1]:
import sys
sys.path.insert(0, "../src")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np

from generation.rve import make_square_composite_rve
from operators.green import build_freq_grid, build_green_operator

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

JAX backend: cpu
Devices: [CpuDevice(id=0)]
X64 enabled: True -> dtype: float64


## Generate the composite RVE

Identical geometry to `lin-elastic_strain.ipynb`: a square-packed 2-fibre RVE, Vf~0.5, 5 um fibre
radius, 10-voxel-thick slab. This is fixed, non-differentiated data -- only the fibre's Young's
modulus below becomes a traced parameter.

In [2]:
phase_np, N, n, L, phi_act = make_square_composite_rve(
    phi=0.5, r_fiber=0.005, spacing=0.0002, N_min=32, nz=10,
)
Nv = int(np.prod(n))
dx = tuple(Li / ni for Li, ni in zip(L, n))
phase = jnp.array(phase_np.reshape(-1))   # 0 = matrix, 1 = fiber

xi_flat = build_freq_grid(n, L)

# Same prescribed macroscopic shear strain as lin-elastic_strain.ipynb.
eps_bar = jnp.array([
    [0.0, 1.0e-3, 0.0],
    [1.0e-3, 0.0, 0.0],
    [0.0, 0.0, 0.0],
])

print("grid n :", n)
print("fiber volume fraction (actual):", phi_act)

grid n : (89, 89, 10)
fiber volume fraction (actual): 0.5010730968312082


## A differentiable CG solve

`cg_diff` is the same algorithm as `solve_elastic`'s inner CG, just restructured around
`jax.lax.scan` with a fixed `maxiter` instead of `jax.lax.while_loop`, freezing the state once
converged so the extra steps are no-ops (identical solution to early-stopping CG, just not free).

`solve_elastic_diff(E_fiber)` rebuilds the matrix material's stiffness field, the reference-medium
Green's operator, and the Lippmann-Schwinger operator from scratch for a given `E_fiber`, then
solves with `cg_diff` -- everything downstream of `E_fiber` is plain differentiable `jax.numpy`,
so both `jax.jvp` and `jax.grad` can trace all the way through.

In [3]:
def cg_diff(A, b, x0, tol, maxiter, M=None):
    M_op = (lambda x: x) if M is None else M
    b_norm = jnp.linalg.norm(b)
    atol = tol * b_norm

    r0 = b - A(x0)
    z0 = M_op(r0)
    p0 = z0
    rz0 = jnp.dot(r0, z0)
    conv0 = jnp.linalg.norm(r0) <= atol

    def step(state, _):
        x, r, p, rz, conv = state
        conv_before = conv  # already converged entering this step -> freeze

        Ap = A(p)
        pAp = jnp.dot(p, Ap)
        alpha = rz / jnp.where(pAp != 0.0, pAp, 1.0)
        x_new = x + alpha * p
        r_new = r - alpha * Ap
        z_new = M_op(r_new)
        rz_new = jnp.dot(r_new, z_new)
        beta = rz_new / jnp.where(rz != 0.0, rz, 1.0)
        p_new = z_new + beta * p
        conv_new = jnp.linalg.norm(r_new) <= atol

        x_out = jnp.where(conv_before, x, x_new)
        r_out = jnp.where(conv_before, r, r_new)
        p_out = jnp.where(conv_before, p, p_new)
        rz_out = jnp.where(conv_before, rz, rz_new)
        conv_out = conv_before | conv_new
        return (x_out, r_out, p_out, rz_out, conv_out), None

    init = (x0, r0, p0, rz0, conv0)
    (x_f, r_f, _, _, converged), _ = jax.lax.scan(step, init, xs=None, length=maxiter)
    return x_f, converged

In [4]:
def solve_elastic_diff(E_fiber, maxiter=100, toler_lin=1e-6):
    nu_matrix, nu_fiber = 0.35, 0.20
    E_matrix = 3.0e3

    def lame(E, nu):
        return E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))

    lam_m, mu_m = lame(E_matrix, nu_matrix)
    lam_f, mu_f = lame(E_fiber, nu_fiber)

    I2 = jnp.eye(3)
    def stiffness(lam, mu):
        return (lam * jnp.einsum('ij,kl->ijkl', I2, I2)
                + mu * (jnp.einsum('ik,jl->ijkl', I2, I2) + jnp.einsum('il,jk->ijkl', I2, I2)))

    C_matrix = stiffness(lam_m, mu_m)
    C_fiber = stiffness(lam_f, mu_f)
    C_field = ((1 - phase)[None, None, None, None, :] * C_matrix[..., None]
               + phase[None, None, None, None, :] * C_fiber[..., None])

    lam0 = 0.5 * (lam_m + lam_f)
    mu0 = 0.5 * (mu_m + mu_f)
    G_glob = build_green_operator(xi_flat, lam0, mu0, scheme="rotated", dx=dx)

    def fft_(x):
        s = x.shape
        return jnp.fft.fftn(x.reshape(s[:-1] + n), axes=(-3, -2, -1)).reshape(s)

    def ifft_(x):
        s = x.shape
        return jnp.fft.ifftn(x.reshape(s[:-1] + n), axes=(-3, -2, -1)).real.reshape(s)

    def A_op(v_flat):
        v = v_flat.reshape(3, 3, Nv)
        Cv = jnp.einsum("ijklm,klm->ijm", C_field, v)
        GCv = jnp.einsum("ijklm,klm->ijm", G_glob, fft_(Cv))
        return ifft_(GCv).reshape(-1)

    eps0 = jnp.ones((3, 3, Nv)) * eps_bar[:, :, None]
    sigma0 = jnp.einsum("ijklm,klm->ijm", C_field, eps0)
    bb = -ifft_(jnp.einsum("ijklm,klm->ijm", G_glob, fft_(sigma0))).reshape(-1)
    x0 = jnp.zeros_like(bb)

    delta_flat, converged = cg_diff(A_op, bb, x0, toler_lin, maxiter)
    delta = delta_flat.reshape(3, 3, Nv)
    eps = eps0 + delta
    sigma = jnp.einsum("ijklm,klm->ijm", C_field, eps)
    return sigma, converged


def loss(E_fiber):
    """Macroscopic shear stress tau_xy as a function of the fibre's E -- the
    same quantity lin-elastic_strain.ipynb prints as `tau_xy (avg)`."""
    sigma, converged = solve_elastic_diff(E_fiber)
    return jnp.mean(sigma[1, 0])

## Compare forward-mode and reverse-mode gradients

`E_fiber = 70e3` MPa, matching the glass fibre in `lin-elastic_strain.ipynb`. There's no simple
closed-form gradient to check against here (unlike the homogeneous sanity checks elsewhere in this
project) -- the composite is heterogeneous, so the "proof" is forward-mode, reverse-mode, and a
central finite difference all agreeing with each other, independently computed three different
ways.

In [5]:
E_fiber0 = 70.0e3

# Reverse-mode: jax.grad (backpropagates through cg_diff's jax.lax.scan).
loss_rev, grad_rev = jax.value_and_grad(loss)(E_fiber0)

# Forward-mode: jax.jvp (propagates a tangent through the same computation).
loss_fwd, grad_fwd = jax.jvp(loss, (E_fiber0,), (1.0,))

# Central finite difference, as a third, independent cross-check.
d_E = 1.0  # MPa
grad_fd = (loss(E_fiber0 + d_E) - loss(E_fiber0 - d_E)) / (2.0 * d_E)

print("loss (tau_xy) at E_fiber=70e3 MPa:", float(loss_rev))
print()
print("d(tau_xy)/d(E_fiber):")
print(f"  reverse-mode (jax.grad)      : {float(grad_rev):.10e}")
print(f"  forward-mode (jax.jvp)       : {float(grad_fwd):.10e}")
print(f"  central finite difference    : {float(grad_fd):.10e}")

rel_diff_rev_fwd = abs(float(grad_rev) - float(grad_fwd)) / abs(float(grad_fwd))
rel_diff_fd = abs(float(grad_fd) - float(grad_fwd)) / abs(float(grad_fwd))
print()
print(f"relative diff (reverse vs forward): {rel_diff_rev_fwd:.3e}")
print(f"relative diff (finite-diff vs forward): {rel_diff_fd:.3e}  (expect ~1e-4 to 1e-6, FD is only 1st-order accurate)")

assert rel_diff_rev_fwd < 1e-6, "forward- and reverse-mode gradients disagree!"
print("\nPASSED -- forward-mode and reverse-mode autodiff agree on d(tau_xy)/d(E_fiber).")

loss (tau_xy) at E_fiber=70e3 MPa: 7.625369073063827

d(tau_xy)/d(E_fiber):
  reverse-mode (jax.grad)      : 1.6463031759e-05
  forward-mode (jax.jvp)       : 1.6463031759e-05
  central finite difference    : 1.6463031757e-05

relative diff (reverse vs forward): 5.762e-15
relative diff (finite-diff vs forward): 1.259e-10  (expect ~1e-4 to 1e-6, FD is only 1st-order accurate)

PASSED -- forward-mode and reverse-mode autodiff agree on d(tau_xy)/d(E_fiber).


## Next steps

- Swap `E_fiber` for any other material parameter (matrix `E`/`nu`, either phase's `nu`) -- nothing
  above is specific to the fibre modulus.
- This is the mechanism [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb)'s and
  [`lin-elastic_mixed-BC.ipynb`](./lin-elastic_mixed-BC.ipynb)'s solves would need to plug into
  for gradient-based inverse calibration (fit `E_fiber`, `E_matrix`, etc. to match a target
  effective stiffness or stress-strain curve via `optax`) -- `cg_diff` here is the missing piece,
  intentionally not merged into `solvers.mechanical.strain_nw_cg` because of the speed cost noted
  above.
- For a heterogeneous problem at production scale, `cg_diff`'s fixed `maxiter` needs to be tuned
  close to the problem's actual convergence (see the note on the
  [Benchmark](https://choROPeNt.github.io/FFTjax/documentation/benchmark) page) rather than left
  generous, since -- unlike `solve_elastic` -- every call here pays for the full budget.